# Module 2: Potential Outcomes Without the Algebra

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

There is a standard way of writing down what an effect is. It has two symbols
and no mathematics beyond subtraction, and once it is in place a surprising
number of arguments become easy to settle.

**About 20 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/YinZhangCISER/Public-Safety-Statistics-Tutorials/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

## 2. Two numbers for every agency

For each agency there are two versions of the same year:

| Symbol | Said out loud |
|---|---|
| **Y(1)** | what the rate was, given that the agency took the training |
| **Y(0)** | what the rate would have been, had it not |

The effect for that agency is **Y(1) minus Y(0)**, or as a percentage,
Y(1) divided by Y(0).

**One of the two is always missing.** For a trained agency you observe Y(1)
and never Y(0). For an untrained agency you observe Y(0) and never Y(1). This
is the whole problem, and it has a name: the fundamental problem of causal
inference.

In [ ]:
print("  what is observed and what is not\n")
for a in TRAINED[:2] + COMPARISON[:2]:
    obs = "Y(1)" if a in TRAINED else "Y(0)"
    mis = "Y(0)" if a in TRAINED else "Y(1)"
    v = cell_rate([a], "after")
    print(f"  {NAME[a]:34s} {obs} = {v:.2f}    {mis} = never observed")

Every method in this series is a way of filling in the missing column. That is
all any of them do.

## 3. Filling in Y(0) from the comparison agencies

The comparison agencies are untrained, so what you observe for them **is**
Y(0). If the two groups would have moved together, their change applies to the
trained agencies too.

In [ ]:
c_mult = cell_rate(COMPARISON, "after") / cell_rate(COMPARISON, "before")
print(f"  the comparison agencies' rate changed by a factor of {c_mult:.3f}")
print(f"  which is {100 * (c_mult - 1):+.1f} percent\n")

rows = []
for a in [x for x in TRAINED if x != "A007"]:
    before = cell_rate([a], "before")
    y1 = cell_rate([a], "after")
    y0 = before * c_mult
    rows.append({"agency": NAME[a],
                 "Y(1), observed": round(y1, 2),
                 "Y(0), constructed": round(y0, 2),
                 "effect": f"{100 * (y1 / y0 - 1):+.1f}%"})
print(f"  the true effect at every one of these is {TRUTH:+.1f} percent\n")
pd.DataFrame(rows).set_index("agency")

Four agencies, one identical true effect of 12 percent, four estimates
spanning from 9 to 21 percent.

**The spread is not telling you that the program worked better at Pinecrest.**
It is telling you that a single agency's rate over a couple of years bounces
around. Reading a story into the ranking would be reading noise.

## 4. Three estimands, and which one you have

Once Y(1) and Y(0) are written down, three different averages become
distinguishable, and reports routinely confuse them.

| Name | The average of Y(1) minus Y(0) over | Answers |
|---|---|---|
| **ATT** | the agencies that **took** the program | did it help the ones who got it |
| **ATU** | the agencies that **did not** | would it have helped the others |
| **ATE** | **all** agencies | would it help on average if everyone took it |

A difference in differences gives you the **ATT**, and only that. It is silent
about what would have happened at the untrained agencies, because nothing in
the data speaks to it.

In [ ]:
keep = [a for a in TRAINED if a != "A007"]
att = 100 * ((cell_rate(keep, "after") / cell_rate(keep, "before")) / c_mult - 1)
print(f"  ATT, the effect on the agencies that took it: {att:+.1f} percent")
print(f"  ATU, the effect on the ones that did not:     not identified")
print(f"  ATE, the effect if everyone took it:          not identified")

**"Not identified" is a finding, not a gap in the analysis.** The trained
agencies had the highest use of force rates in the state; a program that helps
at the top of the distribution may do nothing in the middle. Reporting the ATT
as though it were the ATE is how a program gets rolled out statewide on
evidence that does not cover statewide.

## 5. What the assumption looks like in this notation

The assumption in [Module 1](Module_01_From_It_Went_Down_To_The_Program_Did_It.ipynb)
was "the two groups would have moved together." In potential outcome notation
it is precise:

> **Y(0) for the trained agencies would have changed by the same proportion as
> Y(0) for the comparison agencies.**

It is a statement about a quantity that is never observed, which is why it can
never be verified, only made plausible.

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

pre = f[f["period"] == "before"].copy()
idx = pd.PeriodIndex(pre["year_month"], freq="M")
pre["yr"] = idx.year.values + (idx.month.values - 1) / 12.0

for label, sub in [("the trained agencies, A007 excluded",
                    pre[(pre["trained"] == 1) & (pre["agency_id"] != "A007")]),
                   ("the comparison agencies", pre[pre["trained"] == 0])]:
    z = smf.glm("n_uof ~ yr", sub, family=sm.families.Poisson(),
                offset=np.log(sub["n_arrests"])).fit()
    lo, hi = z.conf_int().loc["yr"]
    print(f"  {label:36s} {100 * (np.exp(z.params['yr']) - 1):+6.2f}% a year  "
          f"[{100 * (np.exp(lo) - 1):+6.2f}, {100 * (np.exp(hi) - 1):+6.2f}]")

The two intervals overlap heavily. That does **not** verify the assumption:
before the program both groups were untreated, so both lines are Y(0), and
agreeing then is not the same as agreeing later.

What it does is make the assumption harder to dismiss. That is the most any
evidence can do for it.

## Exercise

Construct Y(0) a second way, from each trained agency's own pre program trend
rather than from the comparison agencies, and see whether the two agree.

In [ ]:
# Fill in the blank, then run.
RUN = None          # try True

if RUN:
    rows = []
    for a in [x for x in TRAINED if x != "A007"]:
        g = f[f["agency_id"] == a]
        pre_a = g[g["period"] == "before"].copy()
        i1 = pd.PeriodIndex(pre_a["year_month"], freq="M")
        pre_a["yr"] = i1.year.values + (i1.month.values - 1) / 12.0
        z = smf.glm("n_uof ~ yr", pre_a, family=sm.families.Poisson(),
                    offset=np.log(pre_a["n_arrests"])).fit()
        post_a = g[g["period"] == "after"].copy()
        i2 = pd.PeriodIndex(post_a["year_month"], freq="M")
        post_a["yr"] = i2.year.values + (i2.month.values - 1) / 12.0
        y0_trend = 100 * np.exp(z.params["Intercept"]
                                + z.params["yr"] * post_a["yr"]).mean()
        y0_comp = cell_rate([a], "before") * c_mult
        y1 = cell_rate([a], "after")
        rows.append({"agency": NAME[a].split()[0],
                     "Y(0) from comparison": round(y0_comp, 2),
                     "Y(0) from own trend": round(y0_trend, 2),
                     "effect, comparison": f"{100 * (y1 / y0_comp - 1):+.1f}%",
                     "effect, own trend": f"{100 * (y1 / y0_trend - 1):+.1f}%"})
    print(f"  the truth is {TRUTH:+.1f} percent everywhere\n")
    display(pd.DataFrame(rows).set_index("agency"))
else:
    print("Set RUN above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
RUN = True
```

The two constructions of Y(0) disagree, by between 1 and 9 percentage points.

| Agency | From the comparison agencies | From its own pre program trend |
|---|---|---|
| Stonewick | −9.2% | −8.2% |
| Tarnbridge | −17.2% | −23.2% |
| Millgate | −15.5% | −18.4% |
| **Pinecrest** | **−21.0%** | **−12.2%** |

They are built from different information: one uses what happened elsewhere
over the same months, the other uses what happened here over earlier months.
Neither is more correct in general.

Pinecrest is the interesting row, and not because its own trend estimate
happens to land on the truth. **Pinecrest is the campus police agency.** Its
activity follows the academic calendar, collapsing in June and July while
every other agency in the state peaks. Pooling it with weather driven
agencies and assuming they would have moved together is precisely the
assumption that should be doubted here, and the two constructions disagreeing
by 9 points is the data saying so.

**When two defensible counterfactuals agree, that is worth reporting as
evidence.** It does not prove either is right, since both could be wrong in
the same direction. Disagreement is informative in the other direction: at
least one of them is wrong, and it is worth asking which agency is unusual and
why before choosing.

The honest write up names the counterfactual used, gives the estimate under
it, and gives the estimate under the most credible alternative.

</details>

---

**Next:** [Module 3: The Counterfactual You Have to Construct](Module_03_The_Counterfactual_You_Have_To_Construct.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*